# state-dict-load — worked example 2: Strip a 'module.' prefix before loading

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `state-dict-load`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Checkpoints saved from a `DataParallel`/`DistributedDataParallel` model carry a `module.` prefix on every key. To load them into a plain model you remap the state dict by stripping that prefix before calling `load_state_dict`. Key renaming is the standard bridge between checkpoints and a differently-wrapped model.

## Worked solution

We build a small linear model and a checkpoint whose keys are prefixed with `module.` (as DDP would save). We construct a remapped dict by removing the leading `module.` from each key, then `load_state_dict(remapped)` with the default `strict=True`, which now succeeds because every key matches. We print the remapped keys and confirm the model's weight equals the checkpoint's, proving the rename was correct.

In [ ]:
import torch.nn as nn


t.manual_seed(0)
model = nn.Linear(4, 3)
raw_ckpt = {
    'module.weight': t.randn(3, 4),
    'module.bias': t.randn(3),
}
remapped = {k[len('module.'):]: v for k, v in raw_ckpt.items()}
model.load_state_dict(remapped)
print('remapped keys:', sorted(remapped.keys()))
print('weight loaded:', t.equal(model.weight.data, raw_ckpt['module.weight']))